### Hugging Face
- 이미지캡셔닝   :  Salesforce/blip-image-captioning-base
- 텍스트 임베딩  :  sentence-transformers/all-MiniLM-L6-v2
- 텍스트 생성    : distilbert/distilgpt2  (경량 모델)

In [2]:
import os
import fitz
import chromadb
from PIL import Image
from transformers import pipeline,BlipProcessor, BlipForConditionalGeneration
from chromadb.utils import embedding_functions

In [3]:
# BLIP 모델 로드
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [4]:
# 임베딩 모델 로드
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# vectorDB 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db_hf')
collection = chroma_client.get_or_create_collection(name='multimocal_rag_hf',embedding_function=hf_ef)

### 데이터추출 및 오픈소스 임베딩
- PDF 텍스트와 추출된 이미지를 BLIP 모델로 갭셔닝한 뒤, Sentence Transformers임베딩을 사용해서 로컬 vectorDB에 저장

In [ ]:
ptf_path = 'sample_paper.pdf'
if collection.count() == 0:
    doc = fitz.open(ptf_path)
    documents,metadatas,ids = [],[],[]
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추가
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metadatas.append({'type':'text','page':page_num+1})
            ids.append(f'hf_text_page_{page_num+1}')
        # 이미지 캡셔닝
        image_lists = page.get_images(full=True)
        if image_lists:
            xref = image_lists[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            image_filename = f'ht_temp_img.{ext}'
            with open(image_filename, 'wb') as f:
                f.write(image_bytes)
            
            try:
                raw_image = Image.open(image_filename).convert('RGB')
                inputs = processor(raw_image, return_tensor='pt')
                out = model.generate(**inputs,max_new_tokens=50)
                caption = processor.decode(out[0],skip_special_tokens=True)

                documents.append(caption)
                metadatas.append({'type':'image_summary', 'page':page_num+1})
                ids.append(f'hf_image_summary_page_{page_num+1}')
                print(f'페이지 {page_num+1} 이미지 캡셔닝 완료 : {caption}')
            except Exception as e:
                print(f'페이지 {page_num+1} 이미지 캡셔닝 실패 : {e}')
    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)
else:
    print(collection.count())
